In [2]:
#Note** If using gpt model and AzureOpenAI or AzureChatOpenAI (refer: 'Working_with_AzureOpenAI' folder)
import os
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv("/content/.env")

client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-12-01-preview",
)


In [3]:
def get_completion(prompt, deployment="gpt-4.1"):
    response = client.chat.completions.create(
        model=deployment,   #Azure uses DEPLOYMENT name here
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


In [4]:
print(get_completion("Explain transformers in simple terms"))

Sure! Here’s a simple explanation of transformers:

---

**Transformers** are a special kind of computer model that help machines understand and generate language (like answering questions, writing stories, or translating languages).

Imagine you’re reading a sentence. To understand it, you don’t just look at one word at a time—you think about the whole sentence and how the words relate to each other. Transformers do something similar: they look at all the words in a sentence at once and figure out how each word connects to the others.

They use a trick called **attention**. This means, for every word, the transformer pays attention to all the other words to understand the meaning better. For example, in the sentence “The cat sat on the mat because it was tired,” the transformer can figure out that “it” refers to “the cat.”

Because of this, transformers are really good at understanding context and meaning, which makes them great for tasks like chatbots, translation, and summarizing te

In [5]:
#Using Langchain Equivalent
from langchain_openai import AzureChatOpenAI

#from dotenv import load_dotenv
#load_dotenv("/content/.env")

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("API_KEY"),
    api_version="2024-12-01-preview",
    deployment_name="gpt-4.1",
    temperature=0,
)

llm.invoke("Explain transformers in simple terms in 25 words").content


'Transformers are AI models that read and understand text by focusing on important words in context, enabling tasks like translation, summarization, and answering questions.'

In [6]:
print(get_completion("What is 1+1?"))

1 + 1 = 2


In [7]:
## Start by creating an instance of the AzureChatOpenAI class.
chat = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("API_KEY"),
    api_version="2024-12-01-preview",
    deployment_name="gpt-4.1",
    temperature=0,
)

In [8]:
from langchain_core.prompts import ChatPromptTemplate
template_string = """Translate the text that is delimited by triple backticks
into a style that is {style}.
text: ```{text}```"""

prompt_template = ChatPromptTemplate.from_template(template_string)

In [9]:
customer_style = "American English in a casual tone"

customer_email = """
I'm super excited about the new gaming console I bought!
It arrived in just 2 days and I've been playing non-stop.
Totally worth the price!
"""

customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email
)


In [10]:
response = chat.invoke(customer_messages)
print(response.content)

I'm so pumped about the new gaming console I got!  
It showed up in just two days and I haven't stopped playing since.  
Definitely worth every penny!


In [11]:
# Format a new message using a different style.
service_reply = "Hey there, we're glad you're enjoying your new gaming console. Game on!"
service_style_pirate = "a cheerful tone that speaks in Pirate English"
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply
)
print(service_messages[0].content)

Translate the text that is delimited by triple backticks
into a style that is a cheerful tone that speaks in Pirate English.
text: ```Hey there, we're glad you're enjoying your new gaming console. Game on!```


In [12]:
# Generate a response in our new style using the ChatOpenAI instance.
service_response = chat.invoke(service_messages)
print(service_response.content)

Arrr matey! We be mighty glad ye be enjoyin’ yer shiny new gamin’ contraption. Hoist the sails and game on, ye scallywag!


In [ ]:
# Defining how we would like the output from the language model to look like.
desired_output = {
    "gift": False,
    "delivery_days": 2,
    "price_value": "Totally worth the price!"
}

# Create a customer review and a template for extracting information from the review.
customer_review = """\
I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!
"""

review_template = """\
For the following text, extract the following information:
gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.
delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.
price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.
Format the output as JSON with the following keys:
gift
delivery_days
price_value
text: {text}
"""

In [14]:
import langchain_core
import langchain_openai

In [15]:
import langchain
import inspect

In [16]:
print(langchain.__version__)
print(inspect.getfile(langchain))

1.2.3
e:\Lesson_2_demos\venv\lib\site-packages\langchain\__init__.py


In [17]:
#check if model exists
#import langchain.output_parsers as op
#print(dir(op))

In [18]:
#!pip install pydantic

In [19]:
import os
import json
from pydantic import BaseModel


In [20]:
# Step 3: Define Pydantic model for structured output
class ReviewAnalysis(BaseModel):
    gift: bool
    delivery_days: int
    price_value: list[str]

In [21]:
## Start by creating an instance of the AzureChatOpenAI class.
chat = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("API_KEY"),
    api_version="2024-12-01-preview",
    deployment_name="gpt-4.1",
    temperature=0,
)

In [22]:
# Step 5: Prepare prompt template
review_template = """Analyze the following customer review and return a JSON object
matching this structure:

{{
  "gift": true/false,
  "delivery_days": int,
  "price_value": [ ... ]
}}

Review:
{text}
"""

prompt = ChatPromptTemplate.from_template(review_template)

# Step 6: Example review text
customer_review = """
I bought this gaming console as a present for my brother.
It arrived in just 2 days, which was amazing!
The price was a bit high, around $500, but totally worth it.
"""

# Step 7: Format messages
messages = prompt.format_messages(text=customer_review)

# Step 8: Invoke AzureChatOpenAI
response = chat.invoke(messages)

# Step 9: Parse JSON into dictionary
try:
    output_dict = json.loads(response.content)
except json.JSONDecodeError:
    print("Failed to parse JSON. Raw response:")
    print(response.content)
    output_dict = {}
# Step 10: Inspect results
print("Raw dict output:")
print(output_dict)

# Optional: convert to Pydantic model for type safety
try:
    analysis = ReviewAnalysis(**output_dict)
except Exception as e:
    print("Failed to convert to Pydantic model:", e)
    analysis = None

if analysis:
    print("\nPydantic model output:")
    print(analysis)
    print("Gift:", analysis.gift)
    print("Delivery days:", analysis.delivery_days)
    print("Price sentences:", analysis.price_value)

Raw dict output:
{'gift': True, 'delivery_days': 2, 'price_value': ['high', 'worth it']}

Pydantic model output:
gift=True delivery_days=2 price_value=['high', 'worth it']
Gift: True
Delivery days: 2
Price sentences: ['high', 'worth it']


In [23]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence

In [24]:
# Memory as Python list
chat_history = []

def add_to_memory(role: str, content: str):
    chat_history.append({"role": role, "content": content})

# Prompt template that includes memory
prompt_template = ChatPromptTemplate.from_template(
    "{chat_history}\nUser: {input}\nAssistant:"
)

# Function to generate response with memory
def chat_with_memory(user_input: str):
    add_to_memory("user", user_input)
    # Flatten memory into string
    history_str = "\n".join([f"{m['role'].capitalize()}: {m['content']}" for m in chat_history])
    messages = prompt_template.format_messages(chat_history=history_str, input=user_input)
    response = chat.invoke(messages)
    add_to_memory("assistant", response.content)
    return response.content




In [25]:
# Example usage
print(chat_with_memory("Hello! How are you?"))


Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


In [26]:
chat_history

[{'role': 'user', 'content': 'Hello! How are you?'},
 {'role': 'assistant',
  'content': "Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"}]

In [27]:
print(chat_with_memory("so you can help me with anything i ask about?"))

I'll do my best to help you with any questions or topics you have! While I can assist with a wide range of subjects—like science, technology, writing, general advice, and more—there are some limitations. For example, I can't provide medical, legal, or financial advice, and I avoid topics that are unsafe or inappropriate. But for most things, I'm here to help! What would you like to talk about or ask?


In [28]:
chat_history

[{'role': 'user', 'content': 'Hello! How are you?'},
 {'role': 'assistant',
  'content': "Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"},
 {'role': 'user', 'content': 'so you can help me with anything i ask about?'},
 {'role': 'assistant',
  'content': "I'll do my best to help you with any questions or topics you have! While I can assist with a wide range of subjects—like science, technology, writing, general advice, and more—there are some limitations. For example, I can't provide medical, legal, or financial advice, and I avoid topics that are unsafe or inappropriate. But for most things, I'm here to help! What would you like to talk about or ask?"}]

In [29]:
print(chat_with_memory("ok so explain about rockets as you promised and show me the details again"))

Absolutely! Here’s an explanation about rockets, along with some detailed information:

---

### What is a Rocket?

A **rocket** is a vehicle or device that moves by expelling gas or other material at high speed from a nozzle, based on Newton’s Third Law: *For every action, there is an equal and opposite reaction.* Rockets are used for space exploration, launching satellites, fireworks, and even as weapons.

---

### Main Parts of a Rocket

1. **Propellant (Fuel and Oxidizer):**
   - The substance burned to produce thrust.
   - Can be solid or liquid.

2. **Engine/Nozzle:**
   - Where the propellant is burned and hot gases are expelled to create thrust.

3. **Payload:**
   - The cargo the rocket carries (satellites, astronauts, scientific instruments, etc.).

4. **Structure (Airframe):**
   - The body of the rocket, designed to be strong and lightweight.

5. **Fins/Guidance System:**
   - Helps keep the rocket stable and on course.

---

### How Do Rockets Work?

1. **Combustion:** The

In [30]:
print(chat_with_memory("Can you summarize what we talked about so far?"))

Certainly! Here’s a summary of our conversation so far:

1. You greeted me and asked if I could help with anything you ask about.
2. I explained that I can assist with a wide range of topics, but there are some limitations (like medical, legal, or unsafe topics).
3. You asked me to explain rockets and show details.
4. I provided an explanation about rockets, including:
   - What rockets are and how they work
   - Main parts of a rocket (propellant, engine, payload, structure, guidance)
   - Types of rockets (solid, liquid, hybrid)
   - Famous rockets (Saturn V, Space Shuttle, Falcon 9)
   - A fun fact about the history of rockets

Let me know if you want more details or if you have another question!


In [31]:
print(chat_with_memory("explain about rockets again and give details where explaination came from i.e. llm or memory"))

Sure! Here’s an explanation about rockets, and I’ll clarify where the information comes from:

---

### What is a Rocket?

A **rocket** is a vehicle or device that moves by expelling gas or other material at high speed from a nozzle, based on Newton’s Third Law: *For every action, there is an equal and opposite reaction.* Rockets are used for space exploration, launching satellites, fireworks, and even as weapons.

**Source:**  
This explanation comes from my language model (LLM) training, which includes general knowledge from textbooks, encyclopedias, and scientific articles up to June 2024. I do not have personal memory or experience; I generate responses based on patterns in the data I was trained on.

---

### Main Parts of a Rocket

1. **Propellant (Fuel and Oxidizer):**  
   The substance burned to produce thrust. Can be solid or liquid.

2. **Engine/Nozzle:**  
   Where the propellant is burned and hot gases are expelled to create thrust.

3. **Payload:**  
   The cargo the rock

In [32]:
#Multi-step memory aware conversation
from langchain_core.runnables import RunnableSequence

def remember_step(user_input):
    add_to_memory("user", user_input)
    return user_input

def llm_step(user_input):
    history_str = "\n".join([f"{m['role'].capitalize()}: {m['content']}" for m in chat_history])
    messages = prompt_template.format_messages(chat_history=history_str, input=user_input)
    response = chat.invoke(messages)
    add_to_memory("assistant", response.content)
    return response.content

sequence = RunnableSequence(remember_step, llm_step)

# Run sequence
output = sequence.invoke("Hello again!")
print(output)


Hello again! Welcome back. How can I help you today? If you want to continue our previous conversation about rockets or ask about something new, just let me know!


In [33]:
output = sequence.invoke("I have a question ,but not sure if I asked you earlier")
print(output)

No problem! If you’re not sure whether you asked your question earlier, feel free to ask it again. I can reference our current conversation history, but I don’t remember past sessions. Go ahead and ask your question—I'm here to help!


In [34]:
output = sequence.invoke("how far is moon in general from earth")
print(output)

The Moon is, on average, about **384,400 kilometers** (or **238,855 miles**) away from Earth. 

This distance can vary slightly because the Moon’s orbit around Earth is not a perfect circle—it’s an ellipse. So, sometimes the Moon is a bit closer (called **perigee**) and sometimes a bit farther away (called **apogee**):

- **Perigee (closest):** about 363,300 km (225,000 miles)
- **Apogee (farthest):** about 405,500 km (252,000 miles)

But the commonly cited average distance is **384,400 km**.

**Source:**  
This information comes from my language model training, which includes scientific data from sources like NASA and astronomy textbooks.


In [35]:
output = sequence.invoke("I have a question about moon and earth ,but not sure if I asked you earlier")
print(output)

No worries! You can ask your question about the Moon and Earth, even if you’re not sure whether you asked it before. I can reference our current conversation, but I don’t remember past sessions. Please go ahead and ask your question—I'm here to help!


In [36]:
output = sequence.invoke("how far is moon in general from earth,did i ask you this earlier")
print(output)

Yes, you did ask this question earlier in our conversation. Here’s a quick recap:

**Your question:**  
"How far is moon in general from earth?"

**My answer:**  
The Moon is, on average, about **384,400 kilometers** (or **238,855 miles**) away from Earth. This distance varies slightly because the Moon’s orbit is elliptical, so sometimes it’s closer (perigee: ~363,300 km) and sometimes farther (apogee: ~405,500 km), but the average is 384,400 km.

Let me know if you want more details or have another question!


In [37]:
output = sequence.invoke("I have a problem with my printer which is not printing? help me")
print(output)

Of course! Here are some steps to help troubleshoot your printer that isn’t printing:

### 1. **Check the Basics**
- **Power:** Is the printer turned on and plugged in?
- **Connection:** Is it connected to your computer (USB, Wi-Fi, or Bluetooth)?
- **Paper:** Is there paper loaded in the tray?
- **Ink/Toner:** Is there enough ink or toner?

### 2. **Check for Error Messages**
- Look at the printer’s display or your computer for any error messages (like “Paper Jam” or “Low Ink”).

### 3. **Restart Devices**
- Turn off the printer and unplug it for 30 seconds, then plug it back in and turn it on.
- Restart your computer.

### 4. **Check Printer Status**
- On your computer, go to **Control Panel > Devices and Printers** (Windows) or **System Preferences > Printers & Scanners** (Mac).
- Make sure your printer is set as the default and is “Online” (not “Offline” or “Paused”).

### 5. **Clear Print Queue**
- Sometimes jobs get stuck. Open the printer queue and cancel all pending jobs.

### 

In [38]:
output = sequence.invoke("Printer problem is resolved but now i have problem with my keyboard.")
print(output)

I'm glad your printer problem is resolved! Let's troubleshoot your keyboard issue.

**Basic Steps to Fix Keyboard Problems:**

1. **Check Connection:**
   - If it's a wired keyboard, make sure it's securely plugged into your computer.
   - If it's wireless, check the batteries and make sure it's paired or connected.

2. **Restart Your Computer:**
   - Sometimes a simple restart can fix keyboard issues.

3. **Try Another Port or Device:**
   - Plug the keyboard into a different USB port.
   - If possible, try the keyboard on another computer to see if it works there.

4. **Check for Physical Damage:**
   - Look for stuck keys, debris, or visible damage.

5. **Update or Reinstall Drivers:**
   - Go to Device Manager (Windows) or System Preferences (Mac) and check for keyboard driver updates.
   - Uninstall and reinstall the keyboard driver if needed.

6. **Check Keyboard Settings:**
   - Make sure the correct keyboard layout is selected in your operating system.

7. **Test with On-Screen

In [39]:
output = sequence.invoke("my previous problem is back, i thought it was resolved.")
print(output)

I'm sorry to hear your previous problem has returned! To help you effectively, could you clarify which problem you mean—was it the **printer** issue or the **keyboard** issue? 

If you let me know which device is causing trouble and describe the symptoms (e.g., not printing, keys not working, error messages), I can guide you through troubleshooting steps again or suggest new solutions. 

Please provide a bit more detail, and I'll do my best to help!


In [40]:
output = sequence.invoke("previous problem where i wanted to hardcopies of my documents")
print(output)

Thank you for clarifying! It sounds like your **printer problem** has returned and you’re unable to print hardcopies of your documents again.

Let’s go through some troubleshooting steps:

---

### 1. **Check Printer Status**
- Is the printer turned on and connected to your computer (USB, Wi-Fi, etc.)?
- Are there any error lights or messages on the printer display?

### 2. **Check Paper and Ink/Toner**
- Is there paper loaded in the tray?
- Is there enough ink or toner?

### 3. **Check Print Queue**
- On your computer, open the printer queue and see if any jobs are stuck. Cancel all pending jobs and try printing again.

### 4. **Restart Devices**
- Turn off the printer, unplug it for 30 seconds, then plug it back in and turn it on.
- Restart your computer.

### 5. **Check Connections**
- If using USB, try a different cable or port.
- If using Wi-Fi, make sure the printer and computer are on the same network.

### 6. **Update/Reinstall Drivers**
- Go to the printer manufacturer’s websi